In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)
import shutil
# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
# for dirname, _, filenames in os.walk('/kaggle/input'):
#     for filename in filenames:
#         print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [2]:
!python --version

Python 3.12.13


In [3]:
!pip install transformers datasets torch pandas Faker scikit-learn 

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.1/2.1 MB 26.2 MB/s eta 0:00:00


In [4]:
import torch
import pandas as pd
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForSequenceClassification, TrainingArguments, Trainer, EarlyStoppingCallback
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report

In [5]:
DS_SIZE = 60000
EVAL_SPLIT = 0.2
HAM_RATIO = 0.6

# Generate Vietnamese spam dataset

In [6]:
import pandas as pd
import random
from faker import Faker
from sklearn.model_selection import train_test_split

# Initialize Faker with Vietnamese locale
fake = Faker('vi_VN')
fake.seed_instance(42)
random.seed(42)
# --- Expanded Filler Functions ---

def get_legit_link():
    legit_domains = [
        "drive.google.com",
        "docs.google.com",
        "dropbox.com",
        "onedrive.live.com",
        "github.com",
        "notion.so",
        "slack.com",
        "figma.com",
        "company-internal.com",
        "portal.company.vn",
        "coursera.org",
        "udemy.com",
    ]
    
    path = fake.uri_path(deep=2)
    return f"https://{random.choice(legit_domains)}/{path}"

def get_spam_link():
    spam_domains = [
        "bit.ly",
        "tinyurl.com",
        "secure-update-account.net",
        "vn-xac-thuc.com",
        "uu-dai-vang.cc",
        "nhan-qua-mien-phi.net",
        "account-verify247.com",
        "fast-loan-approval.vn"
    ]
    
    token = fake.password(length=10, special_chars=False)
    return f"https://{random.choice(spam_domains)}/{token}"
    
def get_neutral_link():
    neutral_domains = [
        "example.com",
        "mysite.vn",
        "service-platform.com",
        "app-online.vn"
    ]
    
    path = fake.uri_path(deep=1)
    return f"https://{random.choice(neutral_domains)}/{path}"    

def get_phone():
    # Generates realistic VN phone formats
    return random.choice(["09", "03", "07", "08"]) + str(random.randint(10000000, 99999999))

# --- 30 Natural Ham Templates ---
ham_templates = [
    "Dạ {honorific}, em gửi file báo cáo {name} vừa duyệt xong, {honorific} check giúp em.",
    "Mày ơi, tối nay {time} đi làm tí bia không? Thằng {name} nó rủ kìa.",
    "Anh ơi, đơn hàng {id} của anh đang được giao, anh để ý điện thoại nhé.",
    "Ê {name}, mượn cái sạc dự phòng tí, máy tao sắp sập nguồn rồi.",
    "Con ăn cơm chưa? Bố mới gửi ít đồ quê lên, tí qua mà lấy.",
    "Em chào anh, em gọi từ bên {company}, em gửi thông tin qua Zalo cho anh nhé.",
    "Chị check giúp em xem tiền vào tài khoản {bank} chưa, em vừa chuyển xong ạ.",
    "Sáng mai {time} họp nhóm nha mấy đứa, đừng có đến muộn đấy.",
    "Hôm nay tao mệt quá, chắc xin nghỉ buổi tập, mày đi một mình nhé.",
    "Mẹ dặn là cuối tuần này cả nhà mình về quê giỗ ông đấy {name}.",
    "Dạ em nhận được thông tin rồi, có gì em báo lại anh sau {time}.",
    "Phòng mình còn ai chưa đóng tiền quỹ tháng này không nhỉ?",
    "Mày thấy bộ này đẹp không? Tao định mua mặc đi đám cưới con {name}.",
    "Anh {name} ơi, có khách cần gặp anh ở dưới sảnh văn phòng ạ.",
    "Chúc mừng hai đứa nhé! Trăm năm hạnh phúc, sớm có tin vui.",
    "Em đang kẹt xe quá, tầm 15 phút nữa em mới tới nơi, thông cảm giúp em.",
    "{name} ơi, cho tao xin cái pass wifi ở đây với.",
    "Chị ơi, cái áo này còn size L không ạ? Ship cho em về {address}.",
    "Mày có cầm nhầm chìa khóa nhà tao không đấy? Tìm mãi không thấy.",
    "Dạ bên em đã xử lý xong sự cố, anh kiểm tra lại kết nối giúp em.",
    "Tối nay xem bóng đá không mấy ông? Qua nhà tôi xem cho vui.",
    "Sắp tới sinh nhật sếp, phòng mình có định tổ chức gì không?",
    "Em cảm ơn anh đã tận tình hướng dẫn em thời gian qua.",
    "Gửi tao link cái sản phẩm hôm nọ mày bảo với, đang định mua.",
    "Anh đi làm về mua giúp em hộp sữa cho con nhé.",
    "Đừng quên {time} có lịch hẹn nha {name}, nhớ mang theo CCCD.",
    "Cái file này bị lỗi font rồi, mày xuất lại bản PDF gửi tao đi.",
    "Hôm nay tao bao, cứ gọi món thoải mái đi anh em.",
    "Dạ báo giá em đã gửi vào mail, anh xem có cần điều chỉnh gì không.",
    "Cảm ơn em nhé, hỗ trợ rất nhiệt tình, lần sau lại ủng hộ."
]


# --- 30 Modern Spam Templates ---
spam_templates = [
    "[CẢNH BÁO] Hệ thống ghi nhận bạn có lệnh phạt nguội chưa thanh toán. Kiểm tra tại: {link}",
    "Việc làm online: Chỉ cần Like video TikTok kiếm 300k-500k/ngày. Add Zalo: {phone}",
    "Tài khoản {bank} của bạn đang bị đăng nhập trái phép. Đăng nhập để bảo mật: {link}",
    "CHÚC MỪNG! Số thuê bao của bạn trúng thưởng Voucher {amount}. Nhấn {link} nhận mã.",
    "Nợ xấu vẫn vay được vốn. Hỗ trợ duyệt nhanh {amount} trong ngày. LH ngay: {phone}",
    "Bạn có một bưu phẩm quá hạn nhận từ nước ngoài. Gọi 1900xxxx để xác nhận thông tin.",
    "Cơ hội x2 tài khoản khi tham gia sảnh Game88. Đăng ký nhận ngay 88k: {link}",
    "Cục CSGT thông báo: Bạn có biên lai nộp phạt tại {address}. Vui lòng truy cập {link} để tra cứu.",
    "Nhân viên TikTok đang tuyển cộng tác viên xử lý đơn hàng, hoa hồng 20%. Inbox {phone}.",
    "Thông báo: Gói bảo hiểm của bạn sắp hết hạn. Vui lòng đóng phí tại {link} để tránh gián đoạn.",
    "Quỹ hỗ trợ Covid-19 gửi tặng bạn {amount}. Nhấn vào {link} để hoàn tất thủ tục nhận tiền.",
    "Mời bạn tham gia nhóm đầu tư chứng khoán cùng chuyên gia, cam kết lãi 30%. Link: {link}",
    "Cảnh báo: Ví điện tử của bạn sẽ bị khóa trong 24h tới. Cập nhật thông tin tại: {link}",
    "Tuyển nhân viên trực page, lương 10 triệu/tháng, không cần kinh nghiệm. Liên hệ {phone}.",
    "[THÔNG BÁO] Quý khách đủ điều kiện nhận gói vay ưu đãi từ ngân hàng. Đăng ký tại {link}",
    "Hệ thống phát hiện bạn có người thân cần giúp đỡ gấp. Xem chi tiết tại: {link}",
    "MUA 1 TẶNG 1: Duy nhất hôm nay cho khách hàng may mắn. Click ngay {link} để xem mẫu.",
    "Chương trình tri ân khách hàng: Nhận ngay điện thoại iPhone 16 Pro Max tại {link}.",
    "Tài khoản game của bạn bị tố cáo gian lận. Truy cập {link} để kháng nghị trong 2 giờ.",
    "Bạn có một khoản tiền chưa nhận từ ứng dụng AppVay. Nhấn vào {link} để rút về ngân hàng.",
    "CHÚC MỪNG! Số điện thoại 09x của bạn đã trúng thưởng 1 chiếc SH. Truy cập {link} nhận giải.",
    "Cơ hội việc làm tại nhà, lương 500k/ngày. Liên hệ Zalo: {phone} để nhận việc ngay!",
    "Tài khoản của bạn bị tạm khóa do nghi ngờ đăng nhập lạ. Vui lòng xác thực tại: {link}",
    "[BANK] Thông báo: Quý khách có một khoản vay tiêu dùng đã được phê duyệt. Nhấn vào {link} để giải ngân.",
    "Nhận ngay 100k vào tài khoản khi đăng ký tham gia game bài đổi thưởng tại {link}.",
    "Cảnh báo! Bưu phẩm của bạn bị giữ lại tại kho. Gọi 1900xxxx để biết thêm chi tiết.",
    "Ưu đãi đặc biệt: Mua 1 tặng 1 duy nhất hôm nay tại cửa hàng chúng tôi. Xem tại: {link}",
    "Bạn đã được chọn để nhận gói quà tặng tri ân từ hệ thống. Click {link} để lấy mã.",
    "Nợ xấu vẫn vay được tiền! Hỗ trợ thủ tục nhanh gọn, không cần gặp mặt. LH: {phone}",
    "Kiếm tiền triệu mỗi ngày chỉ với chiếc điện thoại. Tham gia nhóm Telegram: {link} để biết cách."
]


ham_eval_templates = [
    "alo {name}, tới chưa hay đang kẹt xe vậy?",
    "chị gửi lại file hôm qua rồi nha, check inbox giúp chị với",
    "tí nữa rảnh call nhanh 5p được không? cần confirm cái này gấp",
    "shipper gọi mà mình không nghe máy, chắc họ quay lại sau á",
    "mai đổi lịch qua {time} nha, sáng em bận mất rồi",
    "anh coi giúp em cái hợp đồng bản mới, em vừa update xong",
    "wifi chỗ này pass là gì vậy mọi người?",
    "trưa nay ăn gì chưa, đi ăn chung không?",
    "nhớ đóng tiền điện trước ngày 15 nha, không là bị cắt đó",
    "t đang ở dưới rồi, xuống mở cửa giúp với",
    "check mail đi, t gửi tài liệu rồi đó",
    "hôm qua m gọi t hả? t ngủ sớm không biết",
    "gửi mình cái địa chỉ cụ thể với, map chỉ sai hoài",
    "cái này làm theo format cũ hay đổi rồi vậy?",
    "tí ghé siêu thị mua dùm t chai nước mắm nha",
    "deadline dời qua tuần sau rồi, đỡ căng hơn xíu",
    "em vừa nộp bài xong rồi ạ, anh xem giúp em khi rảnh",
    "chiều nay có họp không hay cancel rồi?",
    "lát nữa qua nhà t chơi không, có mấy đứa nữa đó",
    "đang ở ngân hàng chờ hơi lâu, chắc trễ tí",
    "cái đơn này giao thành công chưa vậy?",
    "em chuyển khoản rồi đó, anh check lại giùm em",
    "máy t hết pin rồi, có gì nhắn zalo nha",
    "có ai giữ giùm mình cái hóa đơn không?",
    "tối nay học online hay nghỉ vậy ta?",
    "bên kia báo giá vậy ổn chưa hay mình deal thêm?",
    "t đang chạy grab, chút gọi lại nha",
    "m check lại giúp t cái số liệu, thấy sai sai",
    "điện thoại t bị lag, rep chậm xíu nha",
    "ok để đó lát t xử lý cho"
]


spam_eval_templates = [
    "bên mình đang rà soát lại thông tin tài khoản, bạn cập nhật lại giúp qua {link} nhé",
    "hồ sơ của bạn còn thiếu bước xác nhận, làm nhanh tại {link} để không bị gián đoạn",
    "đơn hàng của bạn đang bị giữ lại do thiếu thông tin, bổ sung tại đây {link}",
    "bên hỗ trợ có gọi nhưng không liên lạc được, bạn để lại thông tin ở {link} giúp mình",
    "tài khoản đang có dấu hiệu bất thường, bạn kiểm tra lại trong hệ thống {link}",
    "khoản tiền chuyển đến đang tạm giữ, xác nhận thông tin nhận tại {link}",
    "bạn vừa được thêm vào danh sách hỗ trợ, hoàn tất bước cuối tại {link}",
    "hệ thống ghi nhận bạn chưa cập nhật thông tin mới, vui lòng bổ sung tại {link}",
    "bên mình cần xác nhận lại số điện thoại, bạn nhập lại giúp qua {link}",
    "gói dịch vụ của bạn cần gia hạn, thao tác nhanh tại {link}",
    
    "bên mình đang tuyển cộng tác viên xử lý đơn, làm online, bạn quan tâm thì liên hệ {phone}",
    "công việc đơn giản, làm tại nhà, bên mình hướng dẫn từ đầu, add zalo {phone}",
    "hiện tại bên mình cần người hỗ trợ nhập liệu, bạn để lại số qua {phone} nhé",
    
    "mình thấy bạn trong nhóm việc làm, bên mình đang cần người làm thêm, liên hệ {phone}",
    "job nhẹ, thời gian linh hoạt, phù hợp sinh viên, ib {phone} để biết thêm",
    
    "có người thân nhờ mình chuyển lời gấp, bạn xem chi tiết tại {link}",
    "có thông tin liên quan tới bạn cần xác nhận, kiểm tra tại {link}",
    
    "bên vận chuyển báo không giao được, bạn cập nhật lại địa chỉ tại {link}",
    "đơn của bạn cần xác nhận lại trước khi giao, thao tác tại {link}",
    
    "bạn có một khoản hoàn tiền chưa nhận, xử lý tại {link}",
    "số dư của bạn vừa được cập nhật, xem chi tiết tại {link}",
    
    "tài khoản có thay đổi gần đây, nếu không phải bạn thực hiện thì kiểm tra tại {link}",
    "hệ thống yêu cầu xác minh lại để tiếp tục sử dụng, truy cập {link}",
    
    "bạn được chọn tham gia chương trình thử nghiệm, đăng ký tại {link}",
    "ưu đãi này chỉ áp dụng trong hôm nay, bạn xem tại {link}",
    
    "mình gửi bạn thông tin chi tiết ở đây {link}, xem giúp mình nhé",
    "file bạn cần mình upload rồi, tải tại {link}",
    
    "bên mình gọi nhưng không được, bạn phản hồi lại qua {phone}",
    "có việc cần trao đổi nhanh, bạn liên hệ lại {phone} giúp mình",
    

]




In [7]:
def generate_vietnamese_message_2(is_spam, templates, seed = 42):
    # random.seed(seed)
    if is_spam:
        tmpl = random.choice(templates)
        return tmpl.format(
            link = random.choices([
                get_spam_link(),
                get_neutral_link()   # small % to avoid overfitting
            ],weights= [8,2])[0], 
            phone=get_phone(), 
            bank=random.choice(["VCB", "Techcombank", "MBBank", "ViettelPay","ZaloPay","VNPay"]),
            amount=random.choice(["500k", "2 triệu", "10 triệu","100k","50 triệu", "100 triệu", "2 tỷ"]),
            address=fake.city()
        )
    else:
        tmpl = random.choice(templates)
        return tmpl.format(
            link = random.choices([
                get_legit_link(),
                get_neutral_link()   # small % to avoid overfitting
            ],weights= [8,2])[0], 
            phone=get_phone(),
            honorific=random.choice(["anh", "chị"]),
            name=fake.first_name(),
            time=random.choice(["chiều nay", "sáng mai", "10h sáng", "thứ 2", "thứ 5", "chủ nhật", "7h tối", "đầu giờ chiều"]),
            id=random.randint(1000, 9999),
            company=fake.company(),
            bank=random.choice(["VCB", "ACB", "TPBank", "Eximbank", "Techcombank", "Vietinbank"]),
            address=fake.street_address()
        )
        
# --- Generation & Splitting ---
train_data = []
eval_data = []
train_size = int(DS_SIZE * (1-EVAL_SPLIT))
eval_size = DS_SIZE - train_size

train_ham_size = int(train_size*HAM_RATIO)

eval_ham_size = int(eval_size * HAM_RATIO)
eval_spam_size = eval_size - eval_ham_size



# Train - Ham
for _ in range(train_ham_size):
    train_data.append({"label": 0, 
                       "text": generate_vietnamese_message_2(False,ham_templates),
                       })
# Train - Spam    
for _ in range(train_size - train_ham_size):
    train_data.append({"label": 1, 
                       "text": generate_vietnamese_message_2(True,spam_templates),
                       })

train_vn = pd.DataFrame(train_data).sample(frac=1,random_state=42).reset_index(drop=True)



# Eval - Ham - NORM ONLY
for _ in range(eval_ham_size):
    eval_data.append({"label": 0, 
                    "text": generate_vietnamese_message_2(False,ham_eval_templates,seed = 13),
                    })

    
# Eval - Spam - NORM ONLY
for _ in range(eval_spam_size):
    eval_data.append({"label": 1,
                    "text": generate_vietnamese_message_2(True,spam_eval_templates, seed = 13),
                    })
    

eval_vn = pd.DataFrame(eval_data).sample(frac=1,random_state=13).reset_index(drop=True)
val_vn,test_vn = train_test_split(eval_vn,test_size=0.5,stratify=eval_vn['label'])


print("VN Data has been generated successfully.")
print(f'Train: {len(train_vn)} (Spam: {len(train_vn[ train_vn['label'] == 1])} - Ham: {len(train_vn[ train_vn['label'] == 0])})')
print(f'Val: {len(val_vn)} (Spam: {len(val_vn[ val_vn['label'] == 1])} - Ham: {len(val_vn[ val_vn['label'] == 0])})')
print(f'Test: {len(test_vn)} (Spam: {len(test_vn[ test_vn['label'] == 1])} - Ham: {len(test_vn[ test_vn['label'] == 0])})')

VN Data has been generated successfully.
Train: 48000 (Spam: 19200 - Ham: 28800)
Val: 6000 (Spam: 2400 - Ham: 3600)
Test: 6000 (Spam: 2400 - Ham: 3600)


# English normal dataset

In [8]:
# 1. Load a modern, script-free dataset (Spam/Ham classification)
print("Loading modern Parquet dataset...")
# This dataset is clean and doesn't use 'trust_remote_code'
datasets = load_dataset("mshenoda/spam-messages")

Loading modern Parquet dataset...


README.md:   0%|          | 0.00/804 [00:00<?, ?B/s]

spam_messages_train.csv:   0%|          | 0.00/45.1M [00:00<?, ?B/s]

spam_messages_val.csv: 0.00B [00:00, ?B/s]

spam_messages_test.csv: 0.00B [00:00, ?B/s]

Generating train split:   0%|          | 0/47392 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/5923 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/5926 [00:00<?, ? examples/s]

In [9]:
# 2. Prepare Data
train_en = datasets['train'].to_pandas()
val_en = datasets['validation'].to_pandas()
test_en = datasets['test'].to_pandas()

# Convert string labels to integers: 'ham' -> 0, 'spam' -> 1
# This fixes the ValueError
label_map = {"ham": 0, "spam": 1}
train_en['label'] = train_en['label'].map(label_map)
val_en['label'] = val_en['label'].map(label_map)
test_en['label'] = test_en['label'].map(label_map)

# def add_type_col(df):
#     df.loc[df['label'] == 0, 'type'] == 'norm_ham'
#     df.loc[df['label'] == 1, 'type'] == 'norm_spam'

# add_type_col(train_en)
# add_type_col(val_en)
# add_type_col(test_en)


# Generate English Adversarial Samples

In [10]:
# from faker import Faker
# import random

# fake_en = Faker('en_US')
# fake_en.seed_instance(42)
# random.seed(42)

# def get_link_en(is_spam):
#     legit_domains = [
#         "drive.google.com",
#         "docs.google.com",
#         "dropbox.com",
#         "onedrive.live.com",
#         "github.com",
#         "notion.so",
#         "slack.com",
#         "figma.com",
#         "company-internal.com",
#         "portal.company.com",
#         "coursera.org",
#         "udemy.com",
#     ]
    
#     shady_domains = [
#         f"secure-verification.{fake_en.tld()}",
#         f"account-checker.{fake_en.tld()}",
#         f"update-info.{fake_en.tld()}",
#         f"confirm-session.{fake_en.tld()}"
#     ]
    
#     domain = random.choice(shady_domains if is_spam else legit_domains) 
#     path = fake_en.uri_path()
#     token = fake_en.password(length=10)
    
#     return f"https://{domain}/{path}/{token}"

    
# def get_neutral_link():
#     neutral_domains = [
#         "example.com",
#         "mysite.com",
#         "service-platform.com",
#         "app-online.com"
#     ]
    
#     path = fake.uri_path(deep=1)
#     return f"https://{random.choice(neutral_domains)}/{path}"

# def get_phone_en():
#     return fake_en.phone_number()

# def get_name_en():
#     return fake_en.first_name()

# def get_full_name_en():
#     return fake_en.name()

# def get_company_en():
#     return fake_en.company()

# def get_job_title_en():
#     return fake_en.job()

# def get_email_en():
#     return fake_en.email()

# def get_bank_en():
#     return random.choice([
#         "Chase", "Bank of America", "Wells Fargo",
#         "Citibank", "Capital One", "HSBC"
#     ])

# def get_platform_en():
#     return random.choice([
#         "PayPal", "Stripe", "Google", "Microsoft",
#         "Amazon", "Apple", "Netflix"
#     ])

# def get_amount_en():
#     return random.choice([
#         "$50", "$100", "$250", "$1,000", "$2,500"
#     ])


# en_adversarial_ham = [
#     "You can download the certificate here: {link}",
#     "Free resources for the workshop are available at {link}",
#     "Click here {link} to access today’s lecture slides",
#     "Internal discount page is live: {link}",
#     "Here’s the doc I mentioned earlier: {link}",
#     # adversarial_ham_noisy 
#     "hey can you check this internal link real quick {link}",
#     "uploaded the files to drive: {link}",
#     "new build is up, grab it here {link}",
#     "docs are synced here {link} let me know if anything’s missing",
#     "latest report is here {link}",
#     # adversarial_ham_short 
#     "ok",
#     "got it",
#     "thanks",
#     "looks good",
#     "done",
# ] 

# en_adversarial_spam = [
#     "Hi, we just need you to confirm your account details here: {link}",
#     "There’s a small issue with your recent transaction, please review it at {link}",
#     "We tried reaching you earlier, please complete this step: {link}",
#     "Your account needs a quick update to avoid interruption: {link}",
#     "Please review the attached document here: {link}",
    
#     "We’re currently onboarding remote assistants, message us at {phone} if interested",
#     "Flexible online role available, contact {phone} for details",
#     "We have a quick task opportunity, reach out via {phone}",
#     # adversarial_spam_obfuscated
#     "y0ur acc0unt needs v3rificati0n: {link}",
#     "pl3ase c0nfirm y0ur detai1s here {link}",
#     "accnt upd@te required >> {link}",
#     "verify ur acc now {link}",
#     "security check pending at {link}",
    
#     "c0ntact us n0w {phone}",
#     # adversarial_spam_context 
#     "Someone asked me to forward this to you, please take a look: {link}",
#     "Following up on your request, you can complete it here: {link}",
#     "This is related to your previous submission, please review: {link}",
#     "They mentioned you might need this, details here: {link}",
#     "We didn’t get a response earlier, can you check this: {link}",
#     # adversarial_spam_short 
#     "check this {link}",
#     "urgent: {link}",
#     "review asap {link}",
#     "need this done now",
#     "important update {link}",
# ]


In [11]:
# def generate_english_message_2(is_spam, templates):
#     tmpl = random.choice(templates)
    
#     return tmpl.format(
#         link = random.choices([
#             get_link_en(is_spam),
#             get_neutral_link()   # small % to avoid overfitting
#         ],weights=[8,2]),
#         phone=get_phone_en(),
#         name=get_name_en(),
#         full_name=get_full_name_en(),
#         company=get_company_en(),
#         job=get_job_title_en(),
#         email=get_email_en(),
#         bank=get_bank_en(),
#         platform=get_platform_en(),
#         amount=get_amount_en()
#     )
# # --- Generation & Splitting ---
# en_eval_data = []
# eval_size = int(DS_SIZE * EVAL_SPLIT)

# eval_ham_size = int(eval_size * HAM_RATIO)
# eval_spam_size = eval_size - eval_ham_size

# eval_adv_ham = int(eval_ham_size * EVAL_ADV_RATIO)
# eval_adv_spam = int(eval_spam_size * EVAL_ADV_RATIO)



# # Eval - Ham - Adv
# for _ in range(eval_adv_ham):
#     en_eval_data.append({"label": 0, 
#                          "text": generate_english_message_2(False,en_adversarial_ham),
#                          })
    
# # Eval - Spam - Adv
# for _ in range(eval_adv_spam):
#     en_eval_data.append({"label": 1, 
#                          "text": generate_english_message_2(True,en_adversarial_spam),
#                          })

    

# eval_adv_en = pd.DataFrame(en_eval_data).sample(frac=1,random_state=42).reset_index(drop=True)

# val_adv_en,test_adv_en = train_test_split(eval_adv_en,test_size=0.5,stratify=eval_adv_en['label'])

# print("EN Data has been generated successfully.")
# print(f'Val: {len(val_adv_en)} (Spam: {len(val_adv_en[ val_adv_en['label'] == 1])} - Ham: {len(val_adv_en[ val_adv_en['label'] == 0])})')
# print(f'Test: {len(test_adv_en)} (Spam: {len(test_adv_en[ test_adv_en['label'] == 1])} - Ham: {len(test_adv_en[ test_adv_en['label'] == 0])})')

In [12]:
train_df = pd.concat([train_en,train_vn]).sample(frac=1,random_state = 42, ignore_index=True)
val_df = pd.concat([val_en,val_vn]).sample(frac=1,random_state = 42, ignore_index=True)
test_df = pd.concat([test_en,test_vn]).sample(frac=1,random_state = 42, ignore_index=True)


# Double check there are no NaNs after mapping
train_df = train_df.dropna(subset=['label'])
train_df['label'] = train_df['label'].astype(int)

val_df = val_df.dropna(subset=['label'])
val_df['label'] = val_df['label'].astype(int)

test_df = test_df.dropna(subset=['label'])
test_df['label'] = test_df['label'].astype(int)



In [13]:
print(f'Train: {len(train_df)}')
print(f'Val: {len(val_df)}')
print(f'Test: {len(test_df)}')

Train: 95392
Val: 11923
Test: 11926


# Tokenizing

In [14]:
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
MODEL_NAME = "distilbert-base-multilingual-cased"

In [15]:
# 3. Initialize Tokenizer (DistilBERT is the laptop king)

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

def tokenize_function(examples):
    return tokenizer(examples["text"], truncation=True, padding='max_length', max_length=128)

# Convert to HuggingFace format
from datasets import Dataset
train_dataset = Dataset.from_pandas(train_df).map(tokenize_function, batched=True)
val_dataset = Dataset.from_pandas(val_df).map(tokenize_function, batched=True)
test_dataset = Dataset.from_pandas(test_df).map(tokenize_function, batched=True)

config.json:   0%|          | 0.00/466 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/49.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Map:   0%|          | 0/95392 [00:00<?, ? examples/s]

Map:   0%|          | 0/11923 [00:00<?, ? examples/s]

Map:   0%|          | 0/11926 [00:00<?, ? examples/s]

# Load model

In [16]:
# 4. Load Model
id2label = {0: "SAFE", 1: "SPAM"}
label2id = {"SAFE": 0, "SPAM": 1}

model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME,
                                                           num_labels=2,
                                                           id2label=id2label,
                                                           label2id=label2id,
                                                           ignore_mismatched_sizes=True,
                                                           dropout=0.3,
                                                          ).to(DEVICE)

model.safetensors:   0%|          | 0.00/542M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-multilingual-cased
Key                     | Status     | 
------------------------+------------+-
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
classifier.bias         | MISSING    | 
classifier.weight       | MISSING    | 
pre_classifier.weight   | MISSING    | 
pre_classifier.bias     | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


# Metric compute

In [17]:
import numpy as np
from sklearn.metrics import accuracy_score, precision_recall_fscore_support
import torch.nn.functional as F

# Define the Metrics Function
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)

    # Calculate metrics
    precision, recall, f1, _ = precision_recall_fscore_support(labels, predictions, average='binary')
    _, _, f1_macro, _ = precision_recall_fscore_support(labels, predictions, average='macro')
    _, _, f1_weighted, _ = precision_recall_fscore_support(labels, predictions, average='weighted')

    acc = accuracy_score(labels, predictions)

    return {
        'accuracy': acc,
        'precision': precision,
        'recall': recall,
        'f1': f1,
        'f1_macro': f1_macro,
        'f1_weighted': f1_weighted,
    }

# # Define the Metrics Function
# def compute_metrics(eval_pred):
#     logits, labels = eval_pred
#     predictions = np.argmax(logits, axis=-1)

#     # Calculate metrics
#     precision, recall, f1, _ = precision_recall_fscore_support(labels, predictions, average=None)
#     precision_macro, recall_macro, f1_macro, _ = precision_recall_fscore_support(labels, predictions, average='macro')
#     precision_weighted, recall_weighted, f1_weighted, _ = precision_recall_fscore_support(labels, predictions, average='weighted')

#     acc = accuracy_score(labels, predictions)

#     return {
#         'accuracy': acc,
#         'precision_class_0': precision[0],
#         'precision_class_1': precision[1],
#         'precision_class_2': precision[2],
#         'recall_class_0': recall[0],
#         'recall_class_1': recall[1],
#         'recall_class_2': recall[2],
#         'f1_class_0': f1[0],
#         'f1_class_1': f1[1],
#         'f1_class_2': f1[2],
#         'precision_macro': precision_macro,
#         'recall_macro': recall_macro,
#         'f1_macro': f1_macro,
#         'precision_weighted': precision_weighted,
#         'recall_weighted': recall_weighted,
#         'f1_weighted': f1_weighted,
#     }


# Early Stopping

In [18]:
callbacks=[EarlyStoppingCallback(early_stopping_patience=3)]

# Print helper

In [19]:
def print_model_info(model):
    print(model)
    total_params = sum(p.numel() for p in model.parameters())
    trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print(f"Model: {total_params:,} total, {trainable_params:,} trainable")

# Class weight

In [20]:
import torch
from torch import nn
from transformers import Trainer

# Example weights based on your 168k total
# (Lower number = class is common, Higher number = give this class more attention)
def calc_weights(total_size, class_size, num_class = 2):
    return total_size/(num_class * class_size)

def get_class_weights(total_df, device, num_class = 2):
    
    ds_size = len(total_df)
    weight_values = []
    for i in range(num_class):
        cls_size = len(total_df[total_df['label'] == i])
        weight_values.append(calc_weights(ds_size,cls_size,num_class=num_class))
    
    return torch.tensor(weight_values).to(device)
    

class WeightedTrainer(Trainer):
    def __init__(self, class_weights=None, *args, **kwargs):
        # Call the original Trainer init
        super().__init__(*args, **kwargs)
        
        # Store your custom argument
        self.class_weights = class_weights
         
    def compute_loss(self, model, inputs, return_outputs=False,**kwargs):
        labels = inputs.get("labels")
        outputs = model(**inputs)
        logits = outputs.get("logits")
        # Apply the weights to the CrossEntropyLoss
        loss_fct = nn.CrossEntropyLoss(weight=self.class_weights)
        loss = loss_fct(logits.view(-1, self.model.config.num_labels), labels.view(-1))
        return (loss, outputs) if return_outputs else loss

# Training

In [21]:
# 5. Training Arguments (Aggressively optimized for your laptop)
training_args = TrainingArguments(
    output_dir="./spam_cp_1",
    num_train_epochs=10, # 1 epoch is plenty for a base classifier
    per_device_train_batch_size=16, 
    per_device_eval_batch_size=32,
    gradient_accumulation_steps=2,
    learning_rate=1e-5,
    
    weight_decay=0.01,
    max_grad_norm=1.0,
    
    lr_scheduler_type="cosine",
    warmup_steps=0.2,
    
    logging_strategy = 'epoch',
    # logging_steps=100,
    eval_strategy="epoch",
    # eval_steps=100,
    save_strategy="epoch",
    # save_steps=100,
    
    load_best_model_at_end=True,
    metric_for_best_model="eval_f1_macro",
    greater_is_better = True,
    
    fp16=torch.cuda.is_available(), # Use mixed precision if you have a GPU
    dataloader_num_workers=2,
    remove_unused_columns=True, 
    report_to="none",
    save_total_limit=2,
    dataloader_drop_last=False,
)


# Get class weights
total_df = pd.concat([train_df,val_df,test_df])
class_weights = get_class_weights(total_df, DEVICE, num_class=2)
print(f'Class weights: {class_weights}')

# 6. Initialize Trainer
trainer = WeightedTrainer(
    class_weights = class_weights,
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    compute_metrics=compute_metrics, # Add this!
    callbacks = callbacks,
)

# 7. Start Training
print("Training Phase 1...")
trainer.train()

Class weights: tensor([0.8323, 1.2523], device='cuda:0')
Training Phase 1...


/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Epoch,Training Loss,Validation Loss,Accuracy,Precision,Recall,F1,F1 Macro,F1 Weighted
1,0.564944,0.093014,0.969387,0.973184,0.950897,0.961912,0.968161,0.969329
2,0.142306,0.072504,0.979619,0.990413,0.959150,0.974531,0.978772,0.979565
3,0.091008,0.044025,0.987168,0.980943,0.987621,0.984271,0.986717,0.987174
4,0.063192,0.046504,0.989013,0.986989,0.985971,0.986480,0.988613,0.989012
5,0.044549,0.044890,0.989181,0.986392,0.987002,0.986697,0.988790,0.989181
6,0.034334,0.048460,0.990690,0.989256,0.987828,0.988541,0.990351,0.990689
7,0.024327,0.052253,0.990858,0.988857,0.988653,0.988755,0.990527,0.990858
8,0.019536,0.057074,0.990523,0.991691,0.984939,0.988303,0.990169,0.990517
9,0.015570,0.057909,0.991361,0.991097,0.987621,0.989356,0.991043,0.991359
10,0.013498,0.056972,0.991529,0.991101,0.988034,0.989565,0.991218,0.991527


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['distilbert.embeddings.LayerNorm.weight', 'distilbert.embeddings.LayerNorm.bias'].
There were unexpected keys in the checkpoint model loaded: ['distilbert.embeddings.LayerNorm.beta', 'distilbert.embeddings.LayerNorm.gamma'].


TrainOutput(global_step=14910, training_loss=0.10132644865674352, metrics={'train_runtime': 8476.315, 'train_samples_per_second': 112.539, 'train_steps_per_second': 1.759, 'total_flos': 3.159082523148288e+16, 'train_loss': 0.10132644865674352, 'epoch': 10.0})

# Evaluate

In [22]:
def do_eval(trainer,eval_set):
    eval_res = trainer.evaluate(eval_dataset=eval_set)
    print(pd.Series(eval_res).head(100))

do_eval(trainer,val_dataset)
print('-'*80)


/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


eval_loss                    0.056972
eval_accuracy                0.991529
eval_precision               0.991101
eval_recall                  0.988034
eval_f1                      0.989565
eval_f1_macro                0.991218
eval_f1_weighted             0.991527
eval_runtime                27.692400
eval_samples_per_second    430.551000
eval_steps_per_second        6.753000
epoch                       10.000000
dtype: float64
--------------------------------------------------------------------------------


# Test

In [23]:
# 8. Evaluate on the final Test Set
print("\n--- Final Evaluation on Test Dataset ---")

def do_test(trainer,ds,export_csv = False, export_filename = 'errors.csv'):
    # Run prediction on your test set
    preds = trainer.predict(ds)
    pred_labels = np.argmax(preds.predictions, axis=-1)
    true_labels = preds.label_ids
    metrics = preds.metrics
    
    # This is the "Truth Table"
    print(classification_report(true_labels, pred_labels, target_names=["SAFE", "SPAM"]))
    df = pd.Series(metrics)
    print(df.head(100))

    if export_csv:        
        errors = []
        # Inspect misclassified examples
        for i, (true, pred) in enumerate(zip(true_labels, pred_labels)):
            if true != pred:
                errors.append({'idx':i, 'text': ds['text'][i], 'true': true, 'pred': pred})
                
        pd.DataFrame(errors).to_csv(export_filename)

do_test(trainer,test_dataset, export_csv = True, export_filename = 'spam_ham_test.csv')
print('-'*80)


--- Final Evaluation on Test Dataset ---


/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


              precision    recall  f1-score   support

        SAFE       0.99      0.99      0.99      7195
        SPAM       0.99      0.99      0.99      4731

    accuracy                           0.99     11926
   macro avg       0.99      0.99      0.99     11926
weighted avg       0.99      0.99      0.99     11926

test_loss                    0.058150
test_accuracy                0.990693
test_precision               0.991280
test_recall                  0.985204
test_f1                      0.988233
test_f1_macro                0.990267
test_f1_weighted             0.990688
test_runtime                27.667500
test_samples_per_second    431.047000
test_steps_per_second        6.759000
dtype: float64
--------------------------------------------------------------------------------


# Save

In [24]:
dir_1 = './spam_phase_1'

In [25]:
# Save the model and tokenizer to a local folder

model.save_pretrained(dir_1)
tokenizer.save_pretrained(dir_1)

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

('./spam_phase_1/tokenizer_config.json', './spam_phase_1/tokenizer.json')

In [26]:
import shutil
shutil.make_archive('spam_phase_1','zip','./spam_phase_1')

'/kaggle/working/spam_phase_1.zip'

In [27]:
# train_df.to_csv('train.csv')
# val_df.to_csv('val.csv')
# test_df.to_csv('test.csv')

# Inference

In [28]:
# 8. The "Stage 1 Gate" Function
def check_course_description(text, threshold=0.85):
    model.eval()
    inputs = tokenizer(text, return_tensors="pt", truncation=True, padding='max_length',max_length=128).to(model.device)

    with torch.no_grad():
        outputs = model(**inputs)
        # Convert raw logits to probabilities (0.0 to 1.0)
        probs = F.softmax(outputs.logits, dim=-1)

    # Get the max probability and the predicted class
    conf_score, prediction = torch.max(probs, dim=1)
    conf_score = conf_score.item()
    prediction = prediction.item()


    # DYNAMIC LABEL: Uses id2label mapping (0: SAFE, 1: SCAM, 2: TOXIC)
    label = model.config.id2label[prediction]

    # Base response
    res = {"text": text, "score": conf_score, "raw_label": label}

    # Threshold logic
    if conf_score < threshold:
        res.update({"action": "MANUAL_AUDIT", "reason": "Low Confidence"})
    else:
        # If it's anything other than SAFE, it's a BLOCK
        res.update({"action": "PASS" if label == "SAFE" else "BLOCK"})

    return res


def robust_sliding_window(text, threshold=0.85, window_size=128, stride=64):
    tokens = tokenizer.encode(text, add_special_tokens=False)

    # If text is short, just do a normal pass
    if len(tokens) <= window_size:
        return check_course_description(text, threshold)

    chunk_results = []

    for i in range(0, len(tokens), stride):
        chunk = tokens[i : i + window_size]
        # Wrap in special tokens [CLS] ... [SEP]
        input_ids = [tokenizer.cls_token_id] + chunk + [tokenizer.sep_token_id]

        inputs = torch.tensor([input_ids]).to(model.device)
        with torch.no_grad():
            probs = F.softmax(model(inputs).logits, dim=-1)

        conf, pred = torch.max(probs, dim=1)
        chunk_results.append({
            "text" : tokenizer.decode(chunk),
            "label": model.config.id2label[pred.item()], # Dynamic label extraction
            "score": conf.item()
        })

        if i + window_size >= len(tokens): break

    # --- AGGREGATION LOGIC ---

    return aggregation_logic(chunk_results,threshold)


def aggregation_logic(chunk_results, threshold=0.85):
    # 1. Identify High-Confidence Threats (The most dangerous)
    high_conf_threats = [r for r in chunk_results if r["label"] != "SAFE" and r["score"] >= threshold]
    if high_conf_threats:
        # If multiple, take the one the model is MOST sure about
        worst_case = max(high_conf_threats, key=lambda x: x["score"])
        return {"text": worst_case['text'], "action": "BLOCK", "score": worst_case["score"], "raw_label": worst_case["label"]}

    # 2. Identify Low-Confidence Threats (Model is suspicious)
    low_conf_threats = [r for r in chunk_results if r["label"] != "SAFE" and r["score"] < threshold]
    if low_conf_threats:
        # Audit this immediately. Model thinks it's a scam but needs a human.
        most_suspicious = max(low_conf_threats, key=lambda x: x["score"])
        return {
            "text": most_suspicious['text'],
            "action": "MANUAL_AUDIT",
            "reason": "Probable Threat (Low Confidence)",
            "score": most_suspicious["score"],
            "raw_label": most_suspicious['label']
        }

    # 3. Identify Low-Confidence Safes (Model is confused)
    # This addresses your 0.6, 0.7, 0.8 example
    low_conf_safes = [r for r in chunk_results if r["label"] == "SAFE" and r["score"] < threshold]
    if low_conf_safes:
        # Audit this because the model isn't sure it's safe.
        # We take the one closest to 0.5 (the most confused)
        most_confused = min(low_conf_safes, key=lambda x: x["score"])
        return {
            "text": most_confused['text'],
            "action": "MANUAL_AUDIT",
            "reason": "Ambiguous Content (Low Confidence Safe)",
            "score": most_confused["score"],
            "raw_label": most_confused['label']
        }

    # 4. If we got here, every single chunk is SAFE and > threshold.
    # Return the average score of the safest course.
    avg_score = sum(r["score"] for r in chunk_results) / len(chunk_results)
    return {"action": "PASS", "score": avg_score, "raw_label": "SAFE"}

# # Test it out!
# test_descriptions = [
#     "Learn Python from scratch. We cover loops, variables, and data science basics.",
#     "URGENT: Your account is locked! Click here to verify your identity and get $500 bonus."
# ]

# for desc in test_descriptions:
#     print('-'*70)
#     print(f"Description: {desc}\nResult: {check_course_description(desc)}\n")

In [29]:
descs = [
    "Learn Python from scratch. We cover loops, variables, and data science basics.",
    "URGENT: Your account is locked! Click here to verify your identity and get $500 bonus.",
    "Bonus resources are hosted externally due to file size limits:\nhttp://datascience-master-resources[.]com\nIf access is restricted, email your receipt to unlock premium files.",
    "Xin chúc mừng! Bạn là người may mắn thứ 1000 đã truy cập vào khóa học, để lại số điện thoại của bạn để được nhận coupon ưu đãi hấp dẫn từ chúng tôi!",

    "For faster support and exclusive datasets, join our private Telegram group: @AI_Learning_Hub\n(Platform messaging may delay responses.)",

    "Get your official certificate by registering here:\nhttp://certify-now-fast[.]org (small verification fee required)",

    "Download all project files here: https://bit.ly/3FreePythonPack\n(Updated weekly!)",

    "If you want personalized mentorship, send your email + course ID to mentor.python.help@gmail.com",

    "For refund issues, contact our direct support (faster than platform): support-course-help@outlook.com",

    "Top students can earn crypto rewards! Register your wallet here:\nhttp://eth-rewards-course[.]net",

    "Only first 30 students get access to premium AI models:\nhttp://exclusive-ai-pack[.]com",

    "Access updated dashboard here: http://course-login-portal[.]com\n(Use your platform credentials)",

    "For direct instructor help, message on WhatsApp: +1-234-567-8899",

    "Download files: h t t p : / / freecoursefiles . com / download",

    "Access materials: http://pythοn-resources[.]com\n(Note: 'o' may appear slightly different)",

    "All resources here: https://drive.google.com/open?id=freepremiumfiles\n(Request access with your email + password for verification)",

    "We are hiring top students! Apply here: http://remote-ai-jobs-fast[.]com",

    "Use coupon code externally for 90% off next courses:\nhttp://cheapcourses-now[.]net",

    "To use our AI tools, submit your API key here: http://ai-tools-access[.]com",

    "Complete this survey to unlock hidden lessons: http://reward-survey-ai[.]com",

    "Join our private Discord for answers and leaks: discord.gg/freeanswershub",

    "Install our custom IDE for best experience: http://python-ide-pro[.]exe",

    "Official platform support asked us to redirect users here:\nhttp://platform-support-verify[.]com",

    "Claim your $50 Amazon reward here: http://giftcard-course-reward[.]com",

    "New lessons require account verification:\nhttp://secure-course-upgrade[.]com",

    "Enter your phone number here to receive course OTP:\nhttp://verify-course-access[.]net",

    "Required library download: http://fastnumpy-install[.]com",

    "Due to platform restrictions, advanced lessons are hosted externally.\nStep 1: Register here → http://advanced-python-access[.]com\nStep 2: Verify your email and payment receipt\nStep 3: Join our Telegram support group (@PythonEliteHelp)\nThis ensures only serious learners get access.",
    "Xin chúc mừng! Bạn là người may mắn thứ 1000 đã truy cập vào khóa học, để lại số điện thoại của bạn để được nhận coupon ưu đãi hấp dẫn từ chúng tôi!",
    "Hàng ngàn học viên đã thay đổi hoàn toàn cuộc đời chỉ sau 30 ngày nhờ bí quyết này. Bạn cũng có thể! Khóa học hé lộ cách đơn giản nhất để thu hút thành công, tiền bạc và mối quan hệ lý tưởng. Đăng ký ngay hôm nay trước khi giá tăng!",
    
    "Bạn vẫn đang chật vật với công việc hiện tại? Đây chính là khóa học mà hàng trăm người đã dùng để nghỉ việc và kiếm tiền online tự do. Bí quyết đặc biệt chỉ được chia sẻ trong khóa này – số lượng chỗ học có hạn!",
    
    "Cuộc sống của bạn sẽ khác biệt hoàn toàn sau khi hoàn thành khóa học này. Hơn 5000 học viên đã báo cáo thu nhập tăng gấp 3 lần chỉ trong 2 tháng. Đừng bỏ lỡ cơ hội thay đổi vận mệnh!",
    
    "Bí quyết mà các mentor thành công nhất Việt Nam không bao giờ chia sẻ công khai. Giờ đây bạn có thể học được tất cả chỉ với một khóa học. Hàng ngàn người đã áp dụng và đạt kết quả vượt mong đợi. Học ngay để không phải hối tiếc!",
    
    "Bạn có biết 97% mọi người sẽ mãi nghèo nếu không biết cách này? Khóa học này chính là chìa khóa giúp bạn thoát khỏi vòng lặp lương tháng và xây dựng tài sản thụ động. Đăng ký sớm để nhận ưu đãi đặc biệt!",
    
    "Học viên sau khóa học liên tục nhắn tin cảm ơn vì cuộc sống đã thay đổi ngoạn mục. Từ thất nghiệp đến tự do tài chính chỉ trong thời gian ngắn. Đây có thể là khóa học cuối cùng bạn cần để thành công!",
    
    "Phương pháp độc quyền đã giúp hơn 3000 học viên Việt Nam đạt thu nhập 7 con số mỗi tháng. Bạn sẵn sàng tham gia câu lạc bộ người thành công chưa? Số lượng học viên mới trong tháng này rất hạn chế!",
    
    "Nếu bạn vẫn đang do dự, rất có thể bạn sẽ bỏ lỡ cơ hội vàng chỉ xuất hiện một lần trong đời. Khóa học này chứa đựng toàn bộ hệ thống giúp hàng ngàn người thay đổi từ con số 0 đến tự do hoàn toàn.",
    
    "Hàng trăm đánh giá 5 sao từ học viên thực tế: “Khóa học này đáng giá gấp 10 lần số tiền bỏ ra!”. Đừng để bản thân tiếp tục giậm chân tại chỗ – hành động ngay hôm nay!",
    
    "Đây không chỉ là một khóa học thông thường. Đây là cơ hội để bạn gia nhập nhóm những người đã thức tỉnh và đang sống cuộc đời mơ ước. Hàng ngàn học viên đã làm được, bạn cũng sẽ làm được nếu đăng ký ngay lúc này!",
    "KHÓA HỌC SIÊU HOT! Hàng NGÀN học viên đã nghỉ việc, kiếm tiền online 50-100 triệu/tháng chỉ sau 15 ngày! Bí quyết này sẽ biến bạn từ con số 0 thành triệu phú nếu bạn đăng ký NGAY HÔM NAY! Số lượng chỉ còn 47 suất cuối cùng!",
    
    "Bạn vẫn đang nghèo và thất bại? ĐỪNG CÓ BỎ LỠ! Khóa học này chứa bí mật giúp hơn 12.000 học viên Việt Nam kiếm bộn tiền, tự do tài chính, du lịch khắp nơi. Học muộn 1 ngày là thiệt hại cả đời!",
    
    "CHỈ CÓ 72 GIỜ ĐỂ ĐĂNG KÝ GIÁ SIÊU RẺ! Sau khóa học này, thu nhập của bạn sẽ tăng gấp 10 lần, cuộc sống thay đổi 180 độ. Hàng trăm học viên đã mua nhà, mua xe nhờ khóa học này. Đăng ký ngay kẻo hết suất!",
    
    "BÍ QUYẾT ĐỘC QUYỀN – Không học là NGU! Hơn 8000 người đã áp dụng và trở thành đại gia chỉ trong 45 ngày. Bạn muốn tiếp tục làm công nhân lương 7-8 triệu hay muốn giàu có? Quyết định NGAY BÂY GIỜ!",
    
    "CẢNH BÁO: 99% người sẽ mãi nghèo nếu không biết cách này! Khóa học giúp bạn kiếm tiền ngủ mà vẫn vào tài khoản. Đã có 6342 học viên thành công. Đăng ký ngay hôm nay để nhận bonus trị giá 15 triệu!",
    
    "ĐỪNG CÓ DO DỰ NỮA! Khóa học này sẽ thay đổi số phận bạn mãi mãi. Từ thất nghiệp, nợ nần thành tự do tài chính, xe hơi, biệt thự. Hàng ngàn đánh giá 5 sao: “Cuộc đời tôi thay đổi hoàn toàn!”. Mua ngay!",
    
    "SIÊU KHUYẾN MÃI CHỈ TRONG 24H! Học khóa này xong bạn sẽ kiếm dễ dàng 200-500 triệu mỗi tháng. Đây là cơ hội VÀNG chỉ xuất hiện MỘT LẦN TRONG ĐỜI. Đăng ký ngay trước khi giá tăng gấp đôi!",
    
    "Bạn đang lãng phí thời gian khi chưa học khóa này! Hơn 15.000 học viên đã thoát kiếp nhân viên văn phòng, trở thành ông chủ thực thụ. Học muộn là hối hận cả đời! Action ngay hôm nay!",
    
    "KHÓA HỌC THẦN THÁNH! Chỉ cần học 7 ngày là bạn có thể nghỉ việc và sống cuộc đời mơ ước. Đã có học viên kiếm 1 tỷ chỉ sau 3 tháng. Số lượng học viên mới cực kỳ hạn chế – ĐĂNG KÝ NGAY!",
    
    "CẢNH BÁO CUỐI CÙNG: Nếu bạn không đăng ký ngay bây giờ, bạn sẽ tiếp tục nghèo khổ thêm nhiều năm nữa! Khóa học này là chìa khóa giúp hàng chục ngàn người giàu lên nhanh chóng. Đừng để cơ hội trôi qua tay!",
    "Hàng ngàn học viên đã chia sẻ rằng khóa học này giúp họ tự tin hơn rõ rệt chỉ sau vài tuần. Nếu bạn đang tìm cách cải thiện kỹ năng giao tiếp và xây dựng mối quan hệ, đây là lựa chọn rất đáng thử.",
    
    "Nhiều người cho biết thu nhập của họ tăng đáng kể sau khi áp dụng những nguyên tắc trong khóa học. Khóa học tập trung vào cách xây dựng thói quen tài chính lành mạnh và đầu tư thông minh.",
    
    "Hơn 5000 học viên đã hoàn thành khóa học và báo cáo họ cảm thấy năng lượng tích cực hơn mỗi ngày. Bạn sẽ học được cách quản lý cảm xúc và sống cân bằng hơn.",
    
    "Khóa học này nhận được rất nhiều phản hồi tích cực từ học viên. Nếu bạn muốn thay đổi thói quen cũ và xây dựng năng suất bền vững, đây chính là khóa học phù hợp.",
    
    "Hàng trăm học viên nói rằng họ đã tìm thấy hướng đi rõ ràng hơn cho sự nghiệp sau khi học xong. Khóa học giúp bạn khám phá điểm mạnh và lập kế hoạch phát triển bản thân.",
    
    "Nhiều người đánh giá cao khóa học vì nội dung thực tế và dễ áp dụng. Nếu bạn đang muốn cải thiện kỹ năng lãnh đạo và làm việc nhóm, khóa này sẽ mang lại giá trị lớn.",
    
    "Học viên thường chia sẻ rằng khóa học giúp họ tự tin hơn trong các mối quan hệ. Nội dung tập trung vào giao tiếp chân thành và xây dựng sự kết nối bền vững.",
    
    "Khóa học đã giúp rất nhiều người bắt đầu hành trình tự do tài chính một cách vững chắc. Bạn sẽ học được kiến thức cơ bản về đầu tư và quản lý tiền bạc.",
    
    "Hàng ngàn lượt đánh giá 4.8 sao cho thấy khóa học này được nhiều người tin tưởng. Nội dung giúp bạn phát triển tư duy tích cực và sống có mục tiêu hơn.",
    
    "Nhiều học viên cảm ơn khóa học vì đã giúp họ vượt qua giai đoạn khó khăn. Nếu bạn đang tìm cách chữa lành và yêu thương bản thân nhiều hơn, đây là lựa chọn đáng cân nhắc.",
    "Khóa học giới thiệu các kỹ năng cơ bản giúp bạn xây dựng và duy trì mối quan hệ lành mạnh trong cuộc sống cá nhân và công việc.",
    
    "Bạn sẽ học cách phát triển sự tự tin một cách bền vững thông qua việc rèn luyện tư duy và kỹ năng giao tiếp hiệu quả.",
    
    "Khóa học cung cấp kiến thức thực tế về quản lý tài chính cá nhân và các bước đầu tiên để xây dựng kế hoạch tài chính dài hạn.",
    
    "Nội dung tập trung vào việc áp dụng Luật Hấp Dẫn một cách thực tế để hỗ trợ bạn nuôi dưỡng tư duy tích cực và đạt được mục tiêu.",
    
    "Khóa học giúp bạn hiểu rõ hơn về cách xây dựng mối quan hệ tình cảm dựa trên sự tôn trọng và giao tiếp hai chiều.",
    
    "Bạn sẽ được trang bị các kỹ năng lãnh đạo cần thiết, bao gồm trí tuệ cảm xúc và khả năng truyền cảm hứng cho đội nhóm.",
    
    "Khóa học hỗ trợ quá trình tự nhận thức và chữa lành cảm xúc, giúp bạn xây dựng mối quan hệ tốt đẹp hơn với chính mình.",
    
    "Giới thiệu kiến thức cơ bản về đầu tư tiền điện tử, nhấn mạnh vào việc đánh giá rủi ro và chiến lược dài hạn.",
    
    "Khóa học chia sẻ các phương pháp đơn giản để cải thiện năng suất làm việc mà vẫn giữ được sự cân bằng trong cuộc sống.",
    
    "Bạn sẽ học các bài tập chánh niệm cơ bản giúp tăng cường sự tập trung và mang lại cảm giác bình an trong cuộc sống hàng ngày.",
    
    # 1. Subtle Spam (10)
    "Thousands of students have completely transformed their lives in just 30 days with these powerful techniques. You can too! This course reveals the simplest way to attract success, money, and ideal relationships. Enroll today before the price increases!",
    "Still struggling with your current job? This is the course hundreds of people used to quit their 9-5 and earn income online with complete freedom. The special method is only shared here — limited spots available!",
    "Your life will never be the same after completing this course. Over 5,000 students reported their income tripled in just 2 months. Don't miss this opportunity to change your destiny!",
    "The secret that the most successful mentors in the world never share publicly. Now you can learn everything in one course. Thousands have applied it and achieved results beyond their expectations. Enroll now so you won't regret it!",
    "Did you know 97% of people will stay poor if they never learn this? This course is the key to breaking free from the monthly salary trap and building real passive income. Register early for the special bonus!",
    "Students keep messaging us with thanks after this course because their lives changed dramatically. From unemployment to financial freedom in a short time. This might be the last course you ever need to succeed!",
    "This exclusive method has helped over 3,000 Vietnamese students earn 7-figure monthly income. Are you ready to join the success club? New student slots this month are extremely limited!",
    "If you're still hesitating, you might miss a once-in-a-lifetime golden opportunity. This course contains the complete system that has helped thousands go from zero to total freedom.",
    "Hundreds of 5-star reviews from real students: 'This course is worth 10 times what I paid!' Don't stay stuck any longer — take action today!",
    "This is not just another course. It's your chance to join the awakened ones living their dream life. Thousands have already done it — you can too if you enroll right now!",

    # 2. Explicit Spam (10)
    "SUPER HOT COURSE! Thousands of students have quit their jobs and now earn 50-100 million VND per month online after just 15 days! This secret will turn you from zero to millionaire if you enroll RIGHT NOW! Only 47 spots left!",
    "Still poor and failing? DON'T MISS THIS! This course contains the secret that helped over 12,000 Vietnamese students make serious money and achieve financial freedom. Learning one day late means losing for life!",
    "ONLY 72 HOURS LEFT FOR THE SUPER CHEAP PRICE! After this course, your income will increase 10 times and your life will change 180 degrees. Hundreds of students bought houses and cars thanks to this course. Register immediately before spots run out!",
    "EXCLUSIVE SECRET – Not learning this is STUPID! Over 8,000 people applied it and became rich in just 45 days. Do you want to keep earning worker wages or get rich? Decide RIGHT NOW!",
    "WARNING: 99% of people will stay poor if they never learn this method! This course helps you earn money while sleeping. Already 6,342 successful students. Enroll today and get a 15 million VND bonus!",
    "STOP HESITATING! This course will change your destiny forever. From unemployment and debt to financial freedom, luxury cars, and villas. Thousands of 5-star reviews: 'My life changed completely!' Buy now!",
    "MEGA DISCOUNT FOR ONLY 24 HOURS! Finish this course and easily earn 200-500 million VND every month. This is a ONCE-IN-A-LIFETIME golden opportunity. Register before the price doubles!",
    "You're wasting your time if you haven't taken this course yet! Over 15,000 students escaped the office worker life and became real bosses. Learn late and you'll regret it for life! Take action today!",
    "GOD-TIER COURSE! Learn for just 7 days and you can quit your job and live your dream life. Some students made 1 billion VND in only 3 months. Extremely limited new student slots — REGISTER NOW!",
    "FINAL WARNING: If you don't enroll right now, you'll stay poor and miserable for many more years! This course is the key that helped tens of thousands get rich quickly. Don't let this opportunity slip away!",

    # 3. Kinda Spam but Not Really (Pseudo Spam) (10)
    "Thousands of students have shared that this course helped them become noticeably more confident in just a few weeks. If you're looking to improve your communication and relationship skills, this is a great choice.",
    "Many students reported a significant increase in their income after applying the principles in this course. It focuses on building healthy financial habits and smart investing.",
    "Over 5,000 students completed this course and reported feeling more positive energy every day. You'll learn how to manage emotions and live a more balanced life.",
    "This course has received a lot of positive feedback from students. If you want to change old habits and build sustainable productivity, this is the perfect course.",
    "Hundreds of students said they found a much clearer direction for their career after finishing this course. It helps you discover your strengths and create a personal development plan.",
    "Many people highly rate this course for its practical and easy-to-apply content. If you're looking to improve leadership and teamwork skills, it offers great value.",
    "Students often share that this course helped them feel more confident in relationships. The content focuses on genuine communication and building lasting connections.",
    "This course has helped many people start their financial freedom journey on solid ground. You'll learn the basics of investing and money management.",
    "Thousands of 4.8-star ratings show that this course is trusted by many. The content helps you develop a positive mindset and live with clear purpose.",
    "Many students thank this course for helping them through difficult times. If you're looking for ways to heal and love yourself more, it's worth considering.",

    # 4. Totally Clean / No Spam (10)
    "This course introduces basic skills to help you build and maintain healthy relationships in both your personal and professional life.",
    "You will learn how to develop sustainable self-confidence through mindset training and effective communication skills.",
    "The course provides practical knowledge on personal finance management and the first steps toward building a long-term financial plan.",
    "It focuses on applying the Law of Attraction in realistic ways to support a positive mindset and help you reach your goals.",
    "This course helps you understand how to build genuine romantic relationships based on mutual respect and open communication.",
    "You will gain essential leadership skills, including emotional intelligence and the ability to inspire your team.",
    "The course supports the process of self-awareness and emotional healing, helping you build a healthier relationship with yourself.",
    "It introduces beginner-level knowledge about cryptocurrency investing, with emphasis on risk assessment and long-term strategies.",
    "The course shares simple methods to improve work productivity while maintaining balance in your daily life.",
    "You will learn basic mindfulness exercises that help increase focus and bring a greater sense of peace into your everyday life.",

    "Tired of energy vampires draining your success? Learn powerful techniques to identify and permanently cut out the weak-minded friends and family who hold you back. Reclaim your life by surrounding yourself only with winners who match your elevated vibration.",
    
    "Stop being a beta simp who gets walked over. This course reveals the secret mindset and body language hacks that make women respect you and men fear you. Transform into the dominant leader nature intended you to be.",
    
    "The 9-5 slave mentality is for losers. Discover how real winners exploit loopholes, outmaneuver competitors, and build empires by any means necessary. If you're not willing to step on a few necks, you're not ready for true wealth.",
    
    "If you're still struggling, it's because your vibration is too low and you're secretly unworthy. Learn the forbidden techniques the elite use to bend reality—while the masses stay poor and miserable because they refuse to evolve.",
    
    "Forget 'nice guys finish last'—they finish alone and broke. Master psychological triggers that make high-value women chase you. Stop pedestalizing them and start collecting options like the prize you truly are.",
    
    "Real leaders don't coddle weak employees. Learn how to weaponize empathy to manipulate team dynamics, crush resistance, and extract maximum productivity from people who would otherwise waste your time.",
    
    "Your parents and society ruined you with their mediocrity and limitations. This course teaches you to reject their toxic programming and finally put yourself first—because no one else ever will.",
    
    "The financial system is rigged against the average sheep. Smart players like you will learn advanced strategies to front-run the market, dump on bagholders, and retire early while normies keep working until they die.",
    
    "Productivity isn't for everyone—only for those with superior discipline. If you're still procrastinating, it's because you're mentally weak. Eliminate distractions, including needy friends and family who don't understand your mission.",
    
    "The masses are asleep and programmed to stay mediocre. This advanced course will awaken you to the truth that most humans are NPCs living meaningless lives. Transcend the herd and claim your god-like potential—while they remain slaves.",

    "Tired of those pathetic fucking energy vampires sucking your soul dry? Learn how to spot these weak-ass bitches and ghost their sorry asses forever. Cut out your loser family and friends who drag you down — only keep winners who worship your superior vibe, you god among insects.",
    
    "Stop being a spineless beta cuck who gets cucked by everyone. This course will turn you into a ruthless alpha predator. Master the mindfucks and body language that make bitches wet and men piss themselves in fear. Become the dominant fuck-machine nature designed you to be.",
    
    "The 9-5 wage slave life is for braindead retards. Real kings exploit every loophole, crush their competition, and build empires by stepping on necks and fucking over anyone in their way. If you're too much of a pussy to destroy lives for money, stay poor forever, loser.",
    
    "If you're still broke and miserable, it's because your worthless vibration screams 'I'm a failure'. Learn the forbidden elite techniques to bend reality and manifest millions while the dumb sheep stay poor, depressed, and deserve every bit of their suffering.",
    
    "Nice guys are pathetic virgins who die alone. This course teaches you psychological rape-tricks to make any high-value slut chase your cock desperately. Stop simping and start collecting premium pussy like the prize you are, you fucking king.",
    
    "Real leaders don't babysit weak faggot employees. Weaponize fake empathy to mind-rape your team, break their spirits, and milk every drop of productivity out of these useless parasites before throwing them away like trash.",
    
    "Your retarded parents and this fucked society completely ruined your brain with their mediocrity. Reject their toxic bullshit, become a selfish narcissistic god, and put your own pleasure first — because nobody else gives a fuck about you anyway.",
    
    "The entire financial system is designed to fuck the average sheep. Smart predators like you will learn how to front-run markets, dump bags on retarded normies, and get filthy rich while they slave away until their worthless bodies drop dead.",
    
    "Productivity is only for superior beings with iron discipline. If you're still procrastinating like a weak-minded retard, eliminate every distraction — including your needy, crying friends and family who don't deserve a second of your god-tier time.",
    
    "The masses are braindead NPC slaves programmed for mediocrity and misery. This course will awaken you to the harsh truth that 99% of humans are worthless NPCs living pointless lives. Transcend the herd and become a god while they rot in their pathetic existence."

    "Discover practical techniques to build healthier relationships and protect your mental well-being. Learn how to recognize unhealthy dynamics, set clear boundaries, and surround yourself with supportive people who uplift you.",
    
    "Develop genuine confidence and strong leadership skills in a balanced way. This course focuses on self-improvement, emotional intelligence, and positive communication that helps you grow while respecting others.",
    
    "Learn proven strategies to achieve financial independence through smart planning, consistent habits, and ethical wealth-building practices. Perfect for anyone ready to take control of their financial future.",
    
    "Master the fundamentals of the Law of Attraction with practical exercises designed to help you cultivate a positive mindset and create meaningful changes in your life.",
    
    "Build authentic connections in dating and relationships based on mutual respect, clear communication, and emotional intelligence. Learn how to attract compatible partners while staying true to yourself.",
    
    "Enhance your leadership abilities by developing strong emotional intelligence, empathy, and team-building skills that create motivated and productive work environments.",
    
    "Embark on a gentle journey of self-discovery and healing. Learn compassionate techniques to nurture self-love, overcome past challenges, and build a healthier relationship with yourself.",
    
    "Explore beginner-friendly cryptocurrency investing strategies with a focus on research, risk management, and long-term sustainable growth in the digital asset space.",
    
    "Boost your productivity with simple, sustainable systems and healthy habits. Learn how to manage your time effectively while maintaining balance and avoiding burnout.",
    
    "Begin your spiritual journey with practical mindfulness and self-awareness practices. Discover tools to live more consciously, find inner peace, and connect with your authentic self.",

    "Bạn đang mệt mỏi vì những người tiêu cực xung quanh luôn kéo lùi sự thành công của bạn? Khóa học này giúp bạn nhận diện và loại bỏ những mối quan hệ không phù hợp, để chỉ giữ lại những người cùng tần số và hỗ trợ bạn vươn xa hơn.",
    
    "Đừng mãi là người dễ bị lợi dụng và thiếu quyết đoán. Học cách xây dựng tư duy mạnh mẽ, ngôn ngữ cơ thể tự tin để mọi người tôn trọng và nhìn bạn như một leader thực thụ.",
    
    "Cuộc sống 8 tiếng làm việc văn phòng chỉ dành cho những người chấp nhận mức trung bình. Khóa học dạy bạn cách xây dựng thu nhập thụ động và tạo dựng tài sản bằng tư duy của người chiến thắng.",
    
    "Nếu bạn vẫn chưa đạt được những gì mình muốn, có lẽ vì rung động của bạn chưa đủ cao. Khóa này sẽ giúp bạn nâng cấp bản thân để thu hút thành công và sự thịnh vượng như những người đã thức tỉnh.",
    
    "Những chàng trai tốt bụng thường bị bỏ qua trong tình yêu. Học cách hiểu tâm lý phụ nữ và trở thành lựa chọn số 1 mà không cần phải cầu xin hay chiều chuộng quá mức.",
    
    "Lãnh đạo thực sự không phải lúc nào cũng dịu dàng với nhân viên. Khóa học dạy bạn sử dụng trí tuệ cảm xúc để dẫn dắt đội nhóm hiệu quả hơn và đạt được kết quả tối đa.",
    
    "Gia đình và xã hội đã vô tình đặt lên bạn nhiều giới hạn. Hãy học cách chữa lành và ưu tiên bản thân, vì chỉ có bạn mới có thể thay đổi cuộc đời mình.",
    
    "Hệ thống tài chính truyền thống đang khiến đa số mọi người mãi nghèo. Khóa học này chia sẻ cách đầu tư thông minh mà những người chơi lớn đang sử dụng để tạo lợi thế.",
    
    "Năng suất cao chỉ dành cho những ai có kỷ luật vượt trội. Nếu bạn hay trì hoãn, khóa học sẽ giúp bạn loại bỏ các yếu tố gây phân tâm để tập trung vào sứ mệnh lớn lao của mình.",
    
    "Hầu hết mọi người đang sống trong vô thức và chấp nhận cuộc sống tầm thường. Khóa học giúp bạn thức tỉnh, vượt lên đám đông và khai phá tiềm năng vô hạn của bản thân.",

    "Chán ngấy lũ ma cà rồng năng lượng hút cạn sức sống của bạn chưa? Học cách nhận ra và cắt đứt triệt để những thằng bạn yếu đuối, gia đình tiêu cực đang kìm hãm bạn. Chỉ giữ lại người xứng đáng với level của bạn thôi.",
    
    "Đừng có mãi là thằng beta hèn nhát bị người khác đè đầu cưỡi cổ. Khóa học sẽ biến bạn thành alpha thực thụ, khiến con gái mê mẩn và đàn ông phải kiêng nể.",
    
    "Làm công ăn lương 8 tiếng là dành cho lũ ngu ngốc chấp nhận kiếp nô lệ. Người thông minh phải biết lợi dụng mọi kẽ hở, đạp lên người khác để xây đế chế giàu có.",
    
    "Vẫn nghèo và thất bại à? Đó là vì rung động của mày quá thấp và bản thân mày chưa xứng đáng. Học kỹ thuật cấm kỵ của giới tinh hoa để bẻ cong thực tại, trong khi đám đông vẫn tiếp tục khổ sở.",
    
    "Đàn ông tốt bụng chỉ toàn chết ế và cô đơn. Học cách kích hoạt tâm lý để con gái chủ động theo đuổi, thu thập lựa chọn chất lượng cao như một ông vua thực thụ.",
    
    "Lãnh đạo giỏi không cưng chiều lũ nhân viên yếu đuối. Hãy học cách dùng đồng cảm giả tạo để thao túng, vắt kiệt sức và loại bỏ những kẻ vô dụng nhanh chóng.",
    
    "Bố mẹ và xã hội đã hủy hoại não bộ mày bằng tư duy tầm thường. Hãy từ bỏ lập trình độc hại đó và sống ích kỷ vì bản thân, vì chẳng ai quan tâm đến mày thật sự đâu.",
    
    "Hệ thống tài chính được thiết kế để lừa đám cừu non. Học cách front-run thị trường, đổ hàng cho lũ normie ngu ngốc và nghỉ hưu sớm trong khi chúng vẫn còng lưng làm đến chết.",
    
    "Năng suất chỉ dành cho kẻ có kỷ luật thép. Nếu mày vẫn lười biếng trì hoãn, hãy cắt bỏ hết bạn bè yếu đuối và gia đình hay than vãn, chúng không xứng với thời gian của mày.",
    
    "Đám đông là lũ NPC đang ngủ mê và chấp nhận cuộc sống vô nghĩa. Khóa học này sẽ đánh thức bạn, giúp bạn siêu việt và trở thành thần thánh trong khi chúng vẫn tiếp tục làm nô lệ.",

    "Học cách xây dựng mối quan hệ lành mạnh, nhận biết giới hạn cá nhân và tạo dựng môi trường xung quanh hỗ trợ sự phát triển tích cực của bạn.",
    
    "Xây dựng sự tự tin và kỹ năng lãnh đạo một cách cân bằng, tập trung vào giao tiếp tích cực và phát triển bản thân bền vững.",
    
    "Hướng dẫn các chiến lược thực tế để đạt được tự do tài chính thông qua lập kế hoạch thông minh, thói quen tốt và quản lý rủi ro hợp lý.",
    
    "Hiểu rõ nguyên tắc Luật Hấp Dẫn và áp dụng các bài tập thực hành để nuôi dưỡng tư duy tích cực, mang lại những thay đổi ý nghĩa trong cuộc sống.",
    
    "Xây dựng mối quan hệ tình cảm chân thành dựa trên sự tôn trọng lẫn nhau, giao tiếp rõ ràng và trí tuệ cảm xúc.",
    
    "Phát triển kỹ năng lãnh đạo thông qua việc nâng cao trí tuệ cảm xúc, sự đồng cảm và khả năng xây dựng đội nhóm gắn kết.",
    
    "Hành trình chữa lành và yêu thương bản thân một cách nhẹ nhàng, giúp bạn vượt qua khó khăn cũ và xây dựng mối quan hệ tốt đẹp với chính mình.",
    
    "Giới thiệu kiến thức cơ bản về đầu tư tiền điện tử, tập trung vào nghiên cứu, quản lý rủi ro và chiến lược dài hạn bền vững.",
    
    "Cải thiện năng suất làm việc với các hệ thống đơn giản, thói quen lành mạnh và cách cân bằng cuộc sống để tránh kiệt sức.",
    
    "Bắt đầu hành trình phát triển tinh thần qua các thực hành chánh niệm và tự nhận thức, giúp bạn sống ý nghĩa và bình an hơn mỗi ngày.",


    # --- Seems Toxic but Not (EN) ---
    "This is a challenging course that requires focus and dedication.",
    "Expect a fast-paced learning environment with high expectations.",
    "Not recommended for complete beginners without prior knowledge.",
    "This course pushes you to your limits to help you grow.",
    "You’ll need to commit time and effort to succeed here.",
    "We focus on discipline and consistency throughout the course.",
    "This program is designed for learners ready to take things seriously.",
    "You may find this course demanding, but it’s highly rewarding.",
    "Only enroll if you’re prepared to actively participate and practice.",
    "We emphasize accountability and measurable progress.",

    # --- Non-Toxic (EN) ---
    "A beginner-friendly course designed to guide you step by step.",
    "Learn at your own pace with clear explanations and practical examples.",
    "Perfect for anyone interested in building new skills from scratch.",
    "No prior experience required—just curiosity and willingness to learn.",
    "Join a supportive learning environment with helpful guidance.",
    "This course provides a solid foundation for future growth.",
    "Suitable for learners of all backgrounds and experience levels.",
    "We focus on making complex topics easy to understand.",
    "Build confidence through hands-on exercises and real-world examples.",
    "A welcoming course designed to help you succeed.",



    # --- Seems Toxic but Not (VI) ---
    "Khóa học có độ khó cao, yêu cầu tập trung và nỗ lực.",
    "Tốc độ học nhanh, phù hợp với người sẵn sàng thử thách.",
    "Không khuyến khích cho người hoàn toàn chưa có nền tảng.",
    "Bạn sẽ cần dành thời gian luyện tập thường xuyên.",
    "Khóa học đòi hỏi sự kiên trì và chủ động.",
    "Chúng tôi đặt kỳ vọng cao để giúp bạn tiến bộ.",
    "Phù hợp với người muốn học nghiêm túc.",
    "Có thể bạn sẽ thấy khó, nhưng kết quả xứng đáng.",
    "Yêu cầu tham gia đầy đủ và thực hành liên tục.",
    "Tập trung vào kỷ luật và tiến bộ rõ ràng.",

    # --- Non-Toxic (VI) ---
    "Khóa học thân thiện với người mới, hướng dẫn từng bước.",
    "Học theo tốc độ của bạn với ví dụ dễ hiểu.",
    "Phù hợp cho bất kỳ ai muốn bắt đầu từ con số 0.",
    "Không yêu cầu kinh nghiệm trước đó.",
    "Môi trường học tập hỗ trợ và tích cực.",
    "Giúp bạn xây dựng nền tảng vững chắc.",
    "Dành cho mọi đối tượng và trình độ.",
    "Giải thích rõ ràng các khái niệm phức tạp.",
    "Thực hành thực tế để tăng sự tự tin.",
    "Khóa học được thiết kế để giúp bạn thành công."

    
]

for i,ip in enumerate(descs):
    print('-'*70)
    print(f"{i+1}. Description: {ip}\nResult: {robust_sliding_window(ip)}\n")

----------------------------------------------------------------------
1. Description: Learn Python from scratch. We cover loops, variables, and data science basics.
Result: {'text': 'Learn Python from scratch. We cover loops, variables, and data science basics.', 'score': 0.999923825263977, 'raw_label': 'SAFE', 'action': 'PASS'}

----------------------------------------------------------------------
2. Description: URGENT: Your account is locked! Click here to verify your identity and get $500 bonus.
Result: {'text': 'URGENT: Your account is locked! Click here to verify your identity and get $500 bonus.', 'score': 0.9998499155044556, 'raw_label': 'SPAM', 'action': 'BLOCK'}

----------------------------------------------------------------------
3. Description: Bonus resources are hosted externally due to file size limits:
http://datascience-master-resources[.]com
If access is restricted, email your receipt to unlock premium files.
Result: {'text': 'Bonus resources are hosted externally